# Peptide Generation — Model Evaluation & Visualization

Comprehensive comparison of 8 generative architectures for conditional AMP design.  
Metrics: tag adherence, diversity, novelty, AA distribution, sequence quality.

## 0. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import seaborn as sns
from collections import Counter
from scipy.stats import entropy
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

# optional
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('umap-learn not installed — UMAP plots will be skipped')

In [2]:
# ── Poster-quality plot style ──

DARK_BG = '#0D1117'
CARD_BG = '#161B22'
TEXT = '#E6EDF3'
TEXT_DIM = '#8B949E'
GRID = '#21262D'

MODEL_COLORS = {
    'GPT-2':       '#58A6FF',
    'Transformer': '#79C0FF',
    'LSTM':        '#3FB950',
    'CVAE':        '#BC8CF2',
    'Diffusion':   '#F0883E',
    'MaskGIT':     '#F9826C',
    'EBM':         '#F85149',
    'JEPA':        '#56D4DD',
}

plt.rcParams.update({
    'figure.facecolor': DARK_BG,
    'axes.facecolor': CARD_BG,
    'axes.edgecolor': GRID,
    'axes.labelcolor': TEXT,
    'text.color': TEXT,
    'xtick.color': TEXT_DIM,
    'ytick.color': TEXT_DIM,
    'grid.color': GRID,
    'legend.facecolor': CARD_BG,
    'legend.edgecolor': GRID,
    'font.family': 'sans-serif',
    'font.size': 11,
    'figure.dpi': 150,
})

## 1. Load Data

Load the real training data and generated sequences from each model.  
**Edit the paths below** to match your setup.

In [ ]:
# ── EDIT THESE PATHS ──
TRAIN_CSV = 'data/generator/train_core.csv'  # real training data

# For each model, either:
#   (a) provide a CSV of generated sequences in the same format, or
#   (b) paste sequences directly into the dict below

# Option (b): paste your generation outputs here
GENERATED = {
    'GPT-2': [
        'GIKEFFKRIVQRIKDFLRNLVP',
        'SRIPSDTRRYQNLFSLFHTTN',
        'CPTSVYLRGCSPTRCSRSIRN',
        'GVWKDWIKTAGKIAKAAANAVNQ',
        'IRRTPTNTSGLIILVKLIGNLSAGCPTSSFVLL',
    ],
    'Transformer': [
        'GIKEFFKRIVQRIKDFLRNLVP',
        'IRRTPTNTSGLIILVKLIGNLSAGCPTSSFVLL',
        'SRIPSDTRRYQNLFSLFHTTN',
        'CPTSVYLRGCSPTRCSRSIRN',
        'GVWKDWIKTAGKIAKAAANAVNQ',
    ],
    'LSTM': [
        'GIGKFLHSAKKFGKAFVGEIMNS',
        'ISLITLLTLGSGLLNALKGIKG',
        'ISGTTGVAPAILKSMISVKRIR',
        'GFGVFLKAALKGVAKLIPHIISSIQQ',
        'VRPDPQLVFQSGGIGRLVLRVLLVGR',
    ],
    'CVAE': [
        'KSLSHRYLNYFLRGLSSLGF',
        'RFWLQKQIRAVGILGKFQKIFEKLK',
        'GKVWGRFGKLGENLVRGIADIARNVVTGV',
        'GFLSSVCDLVGGARNGLNGVTCLAQRNKGNLHC',
        'WSLLATCFRGICFTRCNSPECRSPTCLSKYSCCPRN',
    ],
    'Diffusion': [
        'GIVGLFGFLLITLPGGVPAGC',
        'TFCVIGIILGVFVAVAITLFL',
        'LRPLNRERHRAGKRPTLRRLS',
        'TPKRARRKVPRRRKLRPRRAS',
        'AAAVHVGFQLIILAFGTKAK',
    ],
    'MaskGIT': [
        'GFCWNVCVYRNGVRVCHRRCN',
        'RRLLLQWLQLRRKLIELLKHL',
        'GFCWHVCKYRNGVRVCYRRCN',
        'GFCWYVCVYRNGVRVCYRRCN',
        'FLLFLLGAPLSVGLRSIRRHN',
    ],
    'EBM': [
        'GGGGGGGGGGGGGGGGGGGG',
        'SCTCSCTCCTCTRTSSSTTS',
        'ALWTTMLKKVGKMAGHAAK',
        'GFCWNVCVYCNGVRVCHRRC',
        'GFCWNVCVYRNGVRVCHRRC',
    ],
    'JEPA': [
        'RTHRHLLSTRPLLSHSTRAL',
        'RLALSALLLAARVLAKVLPL',
        'SRPYYNPPLPPPVLSPPVSP',
        'KDFLAILSRVLLKIIGIGKL',
        'SPFRSLPPTLPPHLHSRLRS',
    ],
}

# If you have more samples in CSVs, load them:
# GENERATED['GPT-2'] = pd.read_csv('generated_gpt2.csv').iloc[:,0].tolist()

MODEL_NAMES = list(GENERATED.keys())
print(f'Models: {len(MODEL_NAMES)}')
for name, seqs in GENERATED.items():
    print(f'  {name:15s}: {len(seqs)} sequences, mean len={np.mean([len(s) for s in seqs]):.1f}')

In [ ]:
# ── Load real training sequences ──
def parse_csv(path):
    df = pd.read_csv(path)
    col = df.columns[0]
    records = []
    for line in df[col]:
        parts = str(line).strip().split()
        tags = {}
        seq_parts = []
        for p in parts:
            if p == '<EOS>': continue
            elif 'LEN' in p: tags['length'] = p
            elif 'CHARGE' in p: tags['charge'] = p
            elif 'HYDRO' in p: tags['hydro'] = p
            elif not p.startswith('<'): seq_parts.append(p)
        seq = ''.join(seq_parts)
        records.append({**tags, 'sequence': seq})
    return pd.DataFrame(records)

real_data = parse_csv(TRAIN_CSV)
real_sequences = real_data['sequence'].tolist()
print(f'Real training sequences: {len(real_sequences)}')
print(f'Length range: {min(len(s) for s in real_sequences)}-{max(len(s) for s in real_sequences)}')

## 2. Physicochemical Property Computation

In [ ]:
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

# Kyte-Doolittle hydrophobicity
KD_SCALE = {
    'A': 1.8, 'R':-4.5, 'N':-3.5, 'D':-3.5, 'C': 2.5,
    'E':-3.5, 'Q':-3.5, 'G':-0.4, 'H':-3.2, 'I': 4.5,
    'L': 3.8, 'K':-3.9, 'M': 1.9, 'F': 2.8, 'P':-1.6,
    'S':-0.8, 'T':-0.7, 'W':-0.9, 'Y':-1.3, 'V': 4.2,
}

# Charge at pH 7
CHARGE_MAP = {aa: 0 for aa in AMINO_ACIDS}
CHARGE_MAP.update({'R': 1, 'K': 1, 'H': 0.1, 'D': -1, 'E': -1})

def compute_properties(seq):
    if len(seq) == 0:
        return {'length': 0, 'charge': 0, 'hydrophobicity': 0, 'net_charge_density': 0}
    charge = sum(CHARGE_MAP.get(aa, 0) for aa in seq)
    hydro = np.mean([KD_SCALE.get(aa, 0) for aa in seq])
    return {
        'length': len(seq),
        'charge': charge,
        'hydrophobicity': hydro,
        'net_charge_density': charge / len(seq),
    }

def properties_df(sequences, label='real'):
    props = [compute_properties(s) for s in sequences]
    df = pd.DataFrame(props)
    df['model'] = label
    df['sequence'] = sequences
    return df

# compute for all
all_dfs = [properties_df(real_sequences, 'Real (Train)')]
for model, seqs in GENERATED.items():
    all_dfs.append(properties_df(seqs, model))

props_all = pd.concat(all_dfs, ignore_index=True)
print(f'Total property records: {len(props_all)}')
props_all.groupby('model')[['length','charge','hydrophobicity']].describe().round(2)

## 3. Property Distribution — Ridgeline Plots

Compare the distribution of length, charge, and hydrophobicity across models.

In [ ]:
def ridgeline_plot(data, column, title, xlabel, ax=None, models=None):
    if models is None:
        models = ['Real (Train)'] + MODEL_NAMES
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    n = len(models)
    overlap = 0.6

    for i, model in enumerate(reversed(models)):
        subset = data[data['model'] == model][column].dropna()
        if len(subset) < 2:
            continue

        color = '#58A6FF' if model == 'Real (Train)' else MODEL_COLORS.get(model, '#888')
        alpha_fill = 0.35 if model == 'Real (Train)' else 0.55

        from scipy.stats import gaussian_kde
        try:
            kde = gaussian_kde(subset, bw_method=0.3)
            x_range = np.linspace(subset.min() - 1, subset.max() + 1, 200)
            density = kde(x_range)
            density = density / density.max() * 0.85
        except:
            continue

        baseline = i * (1 - overlap)
        ax.fill_between(x_range, baseline, baseline + density,
                       color=color, alpha=alpha_fill, zorder=n - i)
        ax.plot(x_range, baseline + density, color=color,
               linewidth=1.5, zorder=n - i + 1)
        ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else x_range[0] - 0.5,
               baseline + 0.15, model, fontsize=8, color=color,
               fontweight='bold', va='center', zorder=100)

    ax.set_yticks([])
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.spines['left'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


fig, axes = plt.subplots(1, 3, figsize=(20, 7))

ridgeline_plot(props_all, 'length', 'Sequence Length', 'Length (AAs)', ax=axes[0])
ridgeline_plot(props_all, 'charge', 'Net Charge', 'Charge (pH 7)', ax=axes[1])
ridgeline_plot(props_all, 'hydrophobicity', 'Mean Hydrophobicity', 'Kyte-Doolittle', ax=axes[2])

plt.suptitle('Property Distributions: Real vs Generated', fontsize=16,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ridgeline_properties.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 4. Radar Charts — Multi-dimensional Model Comparison

In [ ]:
def compute_model_metrics(model_name, sequences, real_seqs):
    """Compute normalized metrics for radar chart."""
    if len(sequences) == 0:
        return {}

    props = [compute_properties(s) for s in sequences]
    lengths = [p['length'] for p in props]
    charges = [p['charge'] for p in props]
    hydros = [p['hydrophobicity'] for p in props]

    # AA diversity: entropy of AA frequency distribution
    all_aa = ''.join(sequences)
    aa_counts = Counter(all_aa)
    aa_freq = np.array([aa_counts.get(aa, 0) for aa in AMINO_ACIDS], dtype=float)
    aa_freq = aa_freq / aa_freq.sum() if aa_freq.sum() > 0 else aa_freq
    aa_entropy = entropy(aa_freq + 1e-10) / np.log(20)  # normalized 0-1

    # Sequence diversity: mean pairwise edit distance / max_len
    if len(sequences) > 1:
        def edit_dist_norm(s1, s2):
            max_l = max(len(s1), len(s2))
            if max_l == 0: return 0
            diffs = sum(1 for a, b in zip(s1, s2) if a != b)
            diffs += abs(len(s1) - len(s2))
            return diffs / max_l
        dists = []
        for i in range(len(sequences)):
            for j in range(i+1, len(sequences)):
                dists.append(edit_dist_norm(sequences[i], sequences[j]))
        seq_diversity = np.mean(dists)
    else:
        seq_diversity = 0

    # Novelty: fraction of generated seqs NOT in training set
    real_set = set(real_seqs)
    novelty = sum(1 for s in sequences if s not in real_set) / len(sequences)

    # Length consistency: 1 - cv(lengths)
    length_cv = np.std(lengths) / (np.mean(lengths) + 1e-10)
    length_consistency = max(0, 1 - length_cv)

    # Charge spread: std of charges (normalized)
    charge_spread = min(1, np.std(charges) / 5)

    return {
        'AA Diversity': aa_entropy,
        'Seq Diversity': min(1, seq_diversity),
        'Novelty': novelty,
        'Length Consistency': length_consistency,
        'Property Spread': charge_spread,
    }


radar_data = {}
for model, seqs in GENERATED.items():
    radar_data[model] = compute_model_metrics(model, seqs, real_sequences)

# display as table
radar_df = pd.DataFrame(radar_data).T
radar_df.round(3)

In [ ]:
def radar_chart(data_dict, title='Model Comparison'):
    categories = list(next(iter(data_dict.values())).keys())
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    ax.set_facecolor(CARD_BG)
    fig.patch.set_facecolor(DARK_BG)

    for model, metrics in data_dict.items():
        values = [metrics[c] for c in categories]
        values += values[:1]
        color = MODEL_COLORS.get(model, '#888')
        ax.plot(angles, values, color=color, linewidth=2, label=model)
        ax.fill(angles, values, color=color, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10, color=TEXT)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=8, color=TEXT_DIM)
    ax.spines['polar'].set_color(GRID)
    ax.grid(color=GRID, linewidth=0.5)

    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9,
             framealpha=0.9)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20, color=TEXT)

    plt.tight_layout()
    plt.savefig('radar_comparison.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()

radar_chart(radar_data, 'Model Quality Radar')

## 5. Amino Acid Frequency — Heatmap per Model

In [ ]:
AA_ORDERED = list('FWYLIMVAGPSTCNQHKRDE')
AA_ORD_IDX = {aa: i for i, aa in enumerate(AA_ORDERED)}

PROP_COLORS = {
    'F': '#EF5350', 'W': '#EF5350', 'Y': '#EF5350', 'L': '#EF5350',
    'I': '#EF5350', 'M': '#EF5350', 'V': '#EF5350',
    'A': '#FF9800', 'G': '#FF9800', 'P': '#FF9800',
    'S': '#66BB6A', 'T': '#66BB6A', 'C': '#66BB6A',
    'N': '#66BB6A', 'Q': '#66BB6A', 'H': '#66BB6A',
    'K': '#42A5F5', 'R': '#42A5F5',
    'D': '#AB47BC', 'E': '#AB47BC',
}


def aa_frequency(sequences):
    """Compute overall AA frequency."""
    all_aa = ''.join(sequences)
    counts = Counter(all_aa)
    total = sum(counts.values())
    return {aa: counts.get(aa, 0) / max(total, 1) for aa in AA_ORDERED}


# Build frequency matrix: rows=AAs, cols=models
models_for_heatmap = ['Real (Train)'] + MODEL_NAMES
freq_matrix = np.zeros((20, len(models_for_heatmap)))

for j, model in enumerate(models_for_heatmap):
    seqs = real_sequences if model == 'Real (Train)' else GENERATED[model]
    freqs = aa_frequency(seqs)
    for i, aa in enumerate(AA_ORDERED):
        freq_matrix[i, j] = freqs[aa]

fig, ax = plt.subplots(figsize=(14, 8))

cmap = LinearSegmentedColormap.from_list('freq', ['#0D1117', '#1A3550', '#2196F3', '#BBDEFB', '#FFFFFF'])
im = ax.imshow(freq_matrix, aspect='auto', cmap=cmap, interpolation='nearest')

ax.set_yticks(range(20))
ax.set_yticklabels(AA_ORDERED, fontsize=11, fontfamily='monospace', fontweight='bold')
for i, aa in enumerate(AA_ORDERED):
    ax.get_yticklabels()[i].set_color(PROP_COLORS.get(aa, TEXT))

ax.set_xticks(range(len(models_for_heatmap)))
ax.set_xticklabels(models_for_heatmap, fontsize=10, rotation=35, ha='right')
for j, model in enumerate(models_for_heatmap):
    color = '#58A6FF' if model == 'Real (Train)' else MODEL_COLORS.get(model, TEXT)
    ax.get_xticklabels()[j].set_color(color)

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Frequency', color=TEXT)
cbar.ax.yaxis.set_tick_params(color=TEXT_DIM)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_DIM)

ax.set_title('Amino Acid Frequency per Model', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Amino Acid', fontsize=12)

plt.tight_layout()
plt.savefig('aa_frequency_heatmap.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 6. Intra-Model Diversity — Pairwise Distance Heatmaps

In [ ]:
def edit_distance(s1, s2):
    """Simple edit distance (Hamming-like for equal length, padded)."""
    max_l = max(len(s1), len(s2))
    if max_l == 0: return 0
    s1 = s1.ljust(max_l, '-')
    s2 = s2.ljust(max_l, '-')
    return sum(a != b for a, b in zip(s1, s2)) / max_l


ncols = 4
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 9))

for idx, model in enumerate(MODEL_NAMES):
    ax = axes[idx // ncols, idx % ncols]
    seqs = GENERATED[model]
    n = len(seqs)
    dist_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist_mat[i, j] = edit_distance(seqs[i], seqs[j])

    color = MODEL_COLORS.get(model, '#888')
    cmap_model = LinearSegmentedColormap.from_list('d', [CARD_BG, color])
    im = ax.imshow(dist_mat, cmap=cmap_model, vmin=0, vmax=1)
    ax.set_title(f'{model}\nmean dist={dist_mat[np.triu_indices(n,1)].mean():.2f}',
               fontsize=10, fontweight='bold', color=color)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels([str(i+1) for i in range(n)], fontsize=8)
    ax.set_yticklabels([str(i+1) for i in range(n)], fontsize=8)

fig.suptitle('Intra-Model Pairwise Sequence Diversity', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('diversity_matrices.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 7. UMAP Embedding — Real vs Generated

In [ ]:
if not HAS_UMAP:
    print('Skipping UMAP — install with: pip install umap-learn')
else:
    def kmer_features(seq, k=2):
        kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
        counts = Counter(kmers)
        all_kmers = sorted(a + b for a in AMINO_ACIDS for b in AMINO_ACIDS)
        vec = np.array([counts.get(km, 0) for km in all_kmers], dtype=float)
        total = vec.sum()
        if total > 0: vec /= total
        return vec

    # subsample real for speed
    real_sub = np.random.choice(real_sequences, min(3000, len(real_sequences)), replace=False)
    X_real = np.array([kmer_features(s) for s in real_sub])

    # all generated
    gen_labels = []
    gen_features = []
    for model, seqs in GENERATED.items():
        for s in seqs:
            gen_features.append(kmer_features(s))
            gen_labels.append(model)
    X_gen = np.array(gen_features)

    # fit UMAP on real, transform generated
    X_all = np.vstack([X_real, X_gen])
    labels_all = ['Real'] * len(X_real) + gen_labels

    reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, metric='cosine', random_state=42)
    emb = reducer.fit_transform(X_all)

    fig, ax = plt.subplots(figsize=(12, 9))

    # real as background
    n_real = len(X_real)
    ax.scatter(emb[:n_real, 0], emb[:n_real, 1], c='#21262D', s=5, alpha=0.3,
              label='Real (Train)', zorder=1)

    # generated per model
    for model in MODEL_NAMES:
        mask = [l == model for l in labels_all]
        mask_idx = [i for i, m in enumerate(mask) if m]
        color = MODEL_COLORS[model]
        ax.scatter(emb[mask_idx, 0], emb[mask_idx, 1], c=color, s=60,
                  label=model, edgecolors='white', linewidth=0.5, zorder=3, alpha=0.9)

    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    ax.set_xlabel('UMAP 1', fontsize=12)
    ax.set_ylabel('UMAP 2', fontsize=12)
    ax.set_title('Sequence Space: Real vs Generated (2-mer UMAP)', fontsize=14,
                fontweight='bold', pad=12)

    plt.tight_layout()
    plt.savefig('umap_real_vs_generated.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()

## 8. Quantitative Metrics — Bar Chart Summary

In [ ]:
# compute KL divergence of AA distribution vs real
real_freqs = aa_frequency(real_sequences)
real_vec = np.array([real_freqs[aa] for aa in AA_ORDERED]) + 1e-10

metrics_table = []
for model, seqs in GENERATED.items():
    m = compute_model_metrics(model, seqs, real_sequences)

    # KL divergence vs real AA distribution
    gen_freqs = aa_frequency(seqs)
    gen_vec = np.array([gen_freqs[aa] for aa in AA_ORDERED]) + 1e-10
    kl_div = entropy(gen_vec, real_vec)

    metrics_table.append({
        'Model': model,
        'AA Diversity': m['AA Diversity'],
        'Seq Diversity': m['Seq Diversity'],
        'Novelty': m['Novelty'],
        'KL vs Real': kl_div,
    })

metrics_df = pd.DataFrame(metrics_table).set_index('Model')
metrics_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

columns = ['AA Diversity', 'Seq Diversity', 'Novelty', 'KL vs Real']
titles = ['AA Diversity (↑)', 'Sequence Diversity (↑)', 'Novelty (↑)', 'KL Divergence vs Real (↓)']

for ax, col, title in zip(axes, columns, titles):
    values = metrics_df[col]
    colors = [MODEL_COLORS[m] for m in values.index]
    bars = ax.barh(range(len(values)), values, color=colors, alpha=0.85, height=0.7)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(values.index, fontsize=9)
    for i, m in enumerate(values.index):
        ax.get_yticklabels()[i].set_color(MODEL_COLORS[m])
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # value labels
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
               f'{val:.3f}', va='center', fontsize=8, color=TEXT_DIM)

fig.suptitle('Quantitative Metrics Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('metrics_bars.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 9. Positional AA Frequency — Real vs Best Generator

In [ ]:
MAX_POS = 25

def build_positional_freq(sequences, max_pos=25):
    counts = np.zeros((20, max_pos), dtype=float)
    totals = np.zeros(max_pos, dtype=float)
    for seq in sequences:
        for i, aa in enumerate(seq[:max_pos]):
            if aa in AA_ORD_IDX:
                counts[AA_ORD_IDX[aa], i] += 1
                totals[i] += 1
    totals[totals == 0] = 1
    return counts / totals[np.newaxis, :]


freq_real = build_positional_freq(real_sequences, MAX_POS)

# pick best model (Transformer or GPT-2)
best_model = 'Transformer'
freq_gen = build_positional_freq(GENERATED[best_model], MAX_POS)

cmap = LinearSegmentedColormap.from_list('freq', ['#0D1117', '#1A3550', '#2196F3', '#BBDEFB'])
vmax = max(freq_real.max(), freq_gen.max())

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6),
                                     gridspec_kw={'width_ratios': [1, 1, 0.4]})

for ax, freq, title in [(ax1, freq_real, 'Real AMPs'), (ax2, freq_gen, f'{best_model} Generated')]:
    im = ax.imshow(freq, aspect='auto', cmap=cmap, interpolation='nearest', vmin=0, vmax=vmax)
    ax.set_yticks(range(20))
    ax.set_yticklabels(AA_ORDERED, fontsize=9, fontfamily='monospace', fontweight='bold')
    for i, aa in enumerate(AA_ORDERED):
        ax.get_yticklabels()[i].set_color(PROP_COLORS.get(aa, TEXT))
    ax.set_xticks(range(0, MAX_POS, 2))
    ax.set_xticklabels([str(i+1) for i in range(0, MAX_POS, 2)], fontsize=8)
    ax.set_xlabel('Position', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')

# difference
diff = freq_gen - freq_real
diff_cmap = LinearSegmentedColormap.from_list('diff', ['#EF5350', '#0D1117', '#4CAF50'])
im3 = ax3.imshow(diff, aspect='auto', cmap=diff_cmap, interpolation='nearest',
                  vmin=-0.15, vmax=0.15)
ax3.set_yticks(range(20))
ax3.set_yticklabels(AA_ORDERED, fontsize=9, fontfamily='monospace')
ax3.set_xticks(range(0, MAX_POS, 4))
ax3.set_xticklabels([str(i+1) for i in range(0, MAX_POS, 4)], fontsize=8)
ax3.set_xlabel('Position', fontsize=11)
ax3.set_title('Difference (Gen - Real)', fontsize=12, fontweight='bold')
fig.colorbar(im3, ax=ax3, shrink=0.8, label='Δ Frequency')

fig.suptitle('Positional Amino Acid Frequency', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('positional_aa_freq.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 10. Training Curves Comparison

Paste your training log values here (or load from log files).

In [ ]:
# ── PASTE YOUR TRAINING LOSSES HERE ──
# format: {model: [(epoch, train_loss, val_loss), ...]}
# You can parse from the log files or enter key checkpoints

training_curves = {
    'Transformer': [
        (1, 2.73, 2.59), (50, 1.84, 1.82), (100, 1.72, 1.77),
        (200, 1.62, 1.73), (300, 1.57, 1.73), (400, 1.54, 1.72),
    ],
    'LSTM': [
        (1, 2.68, 2.55), (30, 1.79, 1.81), (50, 1.72, 1.79),
        (82, 1.66, 1.78), (132, 1.61, 1.80),
    ],
    'Diffusion': [
        (1, 2.82, 2.72), (50, 2.25, 2.20), (100, 2.14, 2.13),
        (200, 2.05, 2.04), (300, 1.98, 1.99), (363, 1.94, 1.94),
    ],
    'MaskGIT': [
        (1, 2.84, 2.75), (50, 2.30, 2.27), (100, 2.23, 2.21),
        (200, 2.17, 2.16), (327, 2.12, 2.13),
    ],
    'EBM (PL)': [
        (1, 2.75, 2.67), (50, 2.07, 2.02), (100, 1.91, 1.89),
        (200, 1.78, 1.77), (268, 1.74, 1.74),
    ],
    'JEPA': [
        (1, 2.78, 2.66), (50, 2.16, 2.13), (100, 2.07, 2.05),
        (200, 1.98, 1.99), (300, 1.92, 1.96), (454, 1.89, 1.94),
    ],
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for model, data in training_curves.items():
    epochs = [d[0] for d in data]
    train = [d[1] for d in data]
    val = [d[2] for d in data]
    base_model = model.replace(' (PL)', '')
    color = MODEL_COLORS.get(base_model, MODEL_COLORS.get(model, '#888'))

    ax1.plot(epochs, train, color=color, linewidth=2, label=model, marker='o', markersize=4)
    ax2.plot(epochs, val, color=color, linewidth=2, label=model, marker='s', markersize=4)

for ax, title in [(ax1, 'Training Loss'), (ax2, 'Validation Loss')]:
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss (CE)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Training Convergence', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 11. Summary — Final Model Scorecard

In [ ]:
# combine everything into a final table
training_info = {
    'GPT-2':       {'Val Loss': 1.72, 'Epochs': 20, 'Time': '~90m', 'Params': '124M'},
    'Transformer': {'Val Loss': 1.72, 'Epochs': 250, 'Time': '~30m', 'Params': '500K'},
    'LSTM':        {'Val Loss': 1.78, 'Epochs': 132, 'Time': '~15m', 'Params': '550K'},
    'CVAE':        {'Val Loss': 1.46, 'Epochs': 158, 'Time': '~20m', 'Params': '885K'},
    'Diffusion':   {'Val Loss': 1.94, 'Epochs': 363, 'Time': '~25m', 'Params': '500K'},
    'MaskGIT':     {'Val Loss': 2.12, 'Epochs': 327, 'Time': '~24m', 'Params': '500K'},
    'EBM':         {'Val Loss': 1.74, 'Epochs': 268, 'Time': '~50m', 'Params': '900K'},
    'JEPA':        {'Val Loss': 1.94, 'Epochs': 454, 'Time': '~42m', 'Params': '1.9M'},
}

summary_rows = []
for model in MODEL_NAMES:
    row = {'Model': model}
    row.update(training_info.get(model, {}))
    row.update(metrics_df.loc[model].to_dict())
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('Model')

# color-coded display
def highlight_best(s):
    if s.name in ['AA Diversity', 'Seq Diversity', 'Novelty']:
        best = s.max()
        return ['font-weight: bold; color: #3FB950' if v == best else '' for v in s]
    elif s.name == 'KL vs Real':
        best = s.min()
        return ['font-weight: bold; color: #3FB950' if v == best else '' for v in s]
    elif s.name == 'Val Loss':
        best = s.min()
        return ['font-weight: bold; color: #3FB950' if v == best else '' for v in s]
    return ['' for _ in s]

summary.style.apply(highlight_best)

## 12. Save All Figures

All figures have been saved as 300 DPI PNGs with dark background.  
Files: `ridgeline_properties.png`, `radar_comparison.png`, `aa_frequency_heatmap.png`,  
`diversity_matrices.png`, `umap_real_vs_generated.png`, `metrics_bars.png`,  
`positional_aa_freq.png`, `training_curves.png`

In [ ]:
print('All visualizations saved!')
print('\nFiles generated:')
import glob
for f in sorted(glob.glob('*.png')):
    print(f'  {f}')